# Notebook 07: Error Analysis and Stakeholder-Facing Interpretability

**Goal:** Explain where the model fails and which features are associated with failure modes.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from src.data_loader import load_isolet_dataset
from src.inference_pipeline import FeatureSelectionInferencePipeline, PipelineConfig

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

FIG_DIR = Path('outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
X, y, _ = load_isolet_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

pipeline = FeatureSelectionInferencePipeline(
    config=PipelineConfig(
        var_threshold=0.0,
        corr_threshold=0.95,
        corr_method='spearman',
        mi_k=140,
        rfe_feat=160,
        rfe_step=20,
        rfe_use_cv=False,
        rfe_min_features=40,
        l1_C=1.0,
        shap_k=100,
        random_state=42,
    )
)
pipeline.fit(X_fit, y_fit, X_val=X_val, y_val=y_val)

X_test_sel = pipeline.transform(X_test)
y_pred = pipeline.model.predict(X_test_sel)
y_proba = pipeline.model.predict_proba(X_test_sel)


## Confusion matrix and class-wise diagnostics


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues')
plt.title('Confusion Matrix (ISOLET)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_analysis_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.head(10)


In [ ]:
analysis_df = X_test_sel.copy()
analysis_df['y_true'] = y_test.values
analysis_df['y_pred'] = y_pred
analysis_df['error'] = (analysis_df['y_true'] != analysis_df['y_pred']).astype(int)
analysis_df['confidence'] = y_proba.max(axis=1)

error_rate = analysis_df['error'].mean()
print(f'Overall misclassification rate: {error_rate:.4f}')

class_error = (
    analysis_df.groupby('y_true')['error']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'error_rate', 'count': 'support'})
    .sort_values('error_rate', ascending=False)
)
class_error.head(10)


## Feature-level error analysis

**Theory:** If a feature behaves differently for misclassified vs correctly classified samples,
it may reveal a weak decision boundary or class overlap region.


In [ ]:
feature_cols = pipeline.selected_features_
error_group = analysis_df[analysis_df['error'] == 1][feature_cols]
correct_group = analysis_df[analysis_df['error'] == 0][feature_cols]

effect_rows = []
for col in feature_cols:
    effect_rows.append(
        {
            'feature': col,
            'mean_abs_diff': abs(error_group[col].mean() - correct_group[col].mean()),
            'std_error': error_group[col].std(),
            'std_correct': correct_group[col].std(),
        }
    )
effect_df = pd.DataFrame(effect_rows).sort_values('mean_abs_diff', ascending=False)

effect_df.head(15)


In [ ]:
plt.figure(figsize=(9, 6))
top_effect = effect_df.head(12).sort_values('mean_abs_diff')
plt.barh(top_effect['feature'], top_effect['mean_abs_diff'], color='darkorange')
plt.title('Top Features Differentiating Errors vs Correct Predictions')
plt.xlabel('Absolute Mean Difference')
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_analysis_feature_differences.png', dpi=300, bbox_inches='tight')
plt.show()


## SHAP view of error regions

We compare SHAP attribution magnitude for error cases versus correct cases.


In [ ]:
explain_subset = X_test_sel.iloc[:250]
explainer = shap.TreeExplainer(pipeline.model, X_test_sel.sample(min(100, len(X_test_sel)), random_state=42))
shap_values = explainer.shap_values(explain_subset, check_additivity=False)

if isinstance(shap_values, list):
    sv = shap_values[0] if len(shap_values) == 1 else shap_values[1]
elif getattr(shap_values, 'ndim', 2) == 3:
    sv = shap_values[:, :, 0]
else:
    sv = shap_values

local_flags = analysis_df.iloc[: len(explain_subset)]['error'].values.astype(bool)
shap_error = np.abs(sv[local_flags]).mean(axis=0) if local_flags.any() else np.zeros(sv.shape[1])
shap_correct = np.abs(sv[~local_flags]).mean(axis=0) if (~local_flags).any() else np.zeros(sv.shape[1])

shap_diff_df = pd.DataFrame(
    {
        'feature': explain_subset.columns,
        'abs_shap_error': shap_error,
        'abs_shap_correct': shap_correct,
        'delta_error_minus_correct': shap_error - shap_correct,
    }
).sort_values('delta_error_minus_correct', ascending=False)

shap_diff_df.head(15)


## Stakeholder interpretation

- Confusion patterns show which letter groups remain difficult.
- Feature-difference and SHAP-difference tables provide concrete talking points for model-risk review.
- This analysis closes the loop from feature selection to accountable error narratives.
